# Segment all pages of the Bd1-Bd6 books into lines

In [3]:
from pathlib import Path
from kraken import blla
from PIL import Image, ImageDraw
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download
import numpy as np
import cv2
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*_ARRAY_API.*')

In [ ]:
VOLUMES = {
    "Bd3": Path("data/Captain_Cook/Images_Bd3"),
    "Bd4": Path("data/Captain_Cook/Images_Bd4"),
}
OUTPUT_DIR = Path("data/Captain_Cook/all_line_crops")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = hf_hub_download(
    repo_id  = "juliozhao/DocLayout-YOLO-DocStructBench",
    filename = "doclayout_yolo_docstructbench_imgsz1024.pt"
)
yolo_model = YOLOv10(model_path)


def preprocess(pil_img):
    img_cv   = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray     = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return Image.fromarray(cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB))


def get_text_block(page_pil, W, H):
    """
    Detect text block with generous margins on all sides.
    Horizontal margin is now a percentage of box width, not a fixed
    pixel value, since text width varies a lot across pages.
    """
    results = yolo_model.predict(
        preprocess(page_pil), imgsz=1024,
        conf=0.1, verbose=False, device="mps"
    )

    if results and results[0].boxes:
        largest = max(
            results[0].boxes,
            key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) *
                          (b.xyxy[0][3]-b.xyxy[0][1])
        )
        x1, y1, x2, y2 = [int(v) for v in largest.xyxy[0]]

        box_w = x2 - x1
        box_h = y2 - y1

        # Extend top to catch missed first lines (12% of height)
        extend_top = int(box_h * 0.12)

        # Extend left/right by 8% of box width — catches clipped
        # first/last words on each line, which YOLO often misses
        extend_x = int(box_w * 0.08)

        x1 = max(0, x1 - extend_x)
        x2 = min(W, x2 + extend_x)
        y1 = max(0, y1 - extend_top)
        y2 = min(H, y2 + 10)

        return x1, y1, x2, y2

    return 0, 0, W, H


def segment_and_save(page_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    page_pil = Image.open(page_path).convert("RGB")
    W, H     = page_pil.size

    bx1, by1, bx2, by2 = get_text_block(page_pil, W, H)
    crop_pil = preprocess(page_pil).crop((bx1, by1, bx2, by2))
    seg      = blla.segment(crop_pil, raise_on_error=False)

    sorted_lines = sorted(
        seg.lines,
        key=lambda l: sum(p[1] for p in l.baseline) / len(l.baseline)
    )

    saved = 0
    for idx, line in enumerate(sorted_lines):
        if not line.boundary:
            continue

        pts = np.array(line.boundary, dtype=np.int32)

        # Convert from crop-local coords to full-page coords
        # then add generous left/right padding to avoid clipping
        # the first/last word on the line
        x1_local = int(pts[:, 0].min())
        x2_local = int(pts[:, 0].max())
        y1_local = int(pts[:, 1].min())
        y2_local = int(pts[:, 1].max())

        line_w = x2_local - x1_local

        # ── KEY FIX: generous horizontal padding ────────────────────
        # 30px fixed + 5% of line width — covers both short and long lines
        pad_x = max(30, int(line_w * 0.05))

        x1 = max(0, x1_local - pad_x) + bx1
        x2 = min(crop_pil.width, x2_local + pad_x) + bx1
        y1 = max(0, y1_local - 20) + by1
        y2 = min(crop_pil.height, y2_local + 8) + by1

        # Clamp to full page bounds
        x1 = max(0, x1)
        x2 = min(W, x2)
        y1 = max(0, y1)
        y2 = min(H, y2)

        if (x2-x1) < 80 or (x2-x1) < (y2-y1)*2:
            continue

        # Polygon mask — also extend polygon points by pad_x
        # so the mask doesn't cut the padded region we just added
        crop = page_pil.crop((x1, y1, x2, y2))
        mask = Image.new("L", crop.size, 0)
        draw = ImageDraw.Draw(mask)

        # Extend the polygon outward by pad_x on x-axis before drawing
        cx = sum(p[0] for p in line.boundary) / len(line.boundary)
        extended_boundary = []
        for px, py in line.boundary:
            # push points away from horizontal center to widen the mask
            dx = pad_x if px >= cx else -pad_x
            extended_boundary.append((px + dx, py))

        shifted = [
            (px + bx1 - x1, py + by1 - y1)
            for px, py in extended_boundary
        ]
        draw.polygon(shifted, fill=255)

        white  = Image.new("RGB", crop.size, (255, 255, 255))
        masked = Image.composite(crop, white, mask)

        masked.save(out_dir / f"L{idx+1:03d}.png")
        saved += 1

    return saved


# ── Run on all volumes ────────────────────────────────────────────────
total = 0
for vol_name, vol_dir in VOLUMES.items():
    pages = sorted(vol_dir.glob("*.jpg")) + sorted(vol_dir.glob("*.JPG"))
    print(f"\n{vol_name}: {len(pages)} pages")
    for page_path in tqdm(pages, desc=vol_name):
        out_dir = OUTPUT_DIR / vol_name / page_path.stem
        n = segment_and_save(page_path, out_dir)
        total += n

print(f"\nTotal lines saved: {total}")


Bd3: 151 pages


Bd3: 100%|██████████| 151/151 [14:06<00:00,  5.60s/it]



Bd4: 165 pages


Bd4: 100%|██████████| 165/165 [16:26<00:00,  5.98s/it]


Total lines saved: 9988


# Transcribe all lines with the saved best model

In [2]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch, json

MODEL_PATH = Path("cc_finetuned_model")
processor  = TrOCRProcessor.from_pretrained(str(MODEL_PATH))
model      = VisionEncoderDecoderModel.from_pretrained(str(MODEL_PATH))
model.to("mps")
model.eval()

all_transcriptions = {}   # {page_id: [line1, line2, ...]}

for vol_dir in sorted(OUTPUT_DIR.iterdir()):
    for page_dir in sorted(vol_dir.iterdir()):
        page_id = f"{vol_dir.name}/{page_dir.name}"
        lines   = []

        for line_img in sorted(page_dir.glob("*.png")):
            img   = Image.open(line_img).convert("RGB")
            px    = processor(img, return_tensors="pt").pixel_values.to("mps")
            with torch.no_grad():
                ids = model.generate(px, num_beams=4,
                                     max_new_tokens=128)
            text = processor.batch_decode(ids, skip_special_tokens=True)[0]
            lines.append(text)

        all_transcriptions[page_id] = lines
        page_text = " ".join(lines)
        print(f"{page_id}: {len(lines)} lines")

# Save full transcriptions
with open("cook_transcriptions.json", "w", encoding="utf-8") as f:
    json.dump(all_transcriptions, f, ensure_ascii=False, indent=2)

print("Saved cook_transcriptions.json")

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Bd3/B3_P_012: 15 lines
Bd3/B3_P_014: 26 lines
Bd3/B3_P_015: 27 lines
Bd3/B3_P_016: 25 lines
Bd3/B3_P_017: 27 lines
Bd3/B3_P_020: 28 lines
Bd3/B3_P_021: 28 lines
Bd3/B3_P_024: 26 lines
Bd3/B3_P_025: 28 lines
Bd3/B3_P_028: 31 lines
Bd3/B3_P_029: 31 lines
Bd3/B3_P_032: 31 lines
Bd3/B3_P_033: 30 lines
Bd3/B3_P_036: 32 lines
Bd3/B3_P_037: 27 lines
Bd3/B3_P_040: 27 lines
Bd3/B3_P_041: 27 lines
Bd3/B3_P_044: 27 lines
Bd3/B3_P_045: 28 lines
Bd3/B3_P_048: 30 lines
Bd3/B3_P_049: 27 lines
Bd3/B3_P_052: 31 lines
Bd3/B3_P_053: 29 lines
Bd3/B3_P_056: 27 lines
Bd3/B3_P_057: 30 lines
Bd3/B3_P_060: 26 lines
Bd3/B3_P_061: 28 lines
Bd3/B3_P_064: 28 lines
Bd3/B3_P_065: 28 lines
Bd3/B3_P_068: 26 lines
Bd3/B3_P_069: 27 lines
Bd3/B3_P_072: 31 lines
Bd3/B3_P_073: 32 lines
Bd3/B3_P_074: 33 lines
Bd3/B3_P_075: 28 lines
Bd3/B3_P_078: 29 lines
Bd3/B3_P_079: 28 lines
Bd3/B3_P_082: 29 lines
Bd3/B3_P_083: 29 lines
Bd3/B3_P_086: 32 lines
Bd3/B3_P_087: 31 lines
Bd3/B3_P_088: 33 lines
Bd3/B3_P_089: 30 lines
Bd3/B3_P_09

# Build RAG system

In [3]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# Load transcriptions
with open("cook_transcriptions.json") as f:
    transcriptions = json.load(f)

# Build flat text per page
pages = {
    page_id: " ".join(lines)
    for page_id, lines in transcriptions.items()
}

# Chunk into passages (~200 words each)
def chunk_text(text, page_id, chunk_size=200):
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append({
            "text"   : chunk,
            "page_id": page_id,
            "chunk"  : i // chunk_size
        })
    return chunks

all_chunks = []
for page_id, text in pages.items():
    all_chunks.extend(chunk_text(text, page_id))

print(f"Total chunks: {len(all_chunks)}")

# Embed and store in ChromaDB
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client   = chromadb.PersistentClient(path="cook_chromadb")
col      = client.get_or_create_collection("cook_journal")

for i, chunk in enumerate(all_chunks):
    embedding = embedder.encode(chunk["text"]).tolist()
    col.add(
        ids        = [f"chunk_{i}"],
        embeddings = [embedding],
        documents  = [chunk["text"]],
        metadatas  = [{"page_id": chunk["page_id"],
                       "chunk"  : chunk["chunk"]}]
    )

print(f"Stored {len(all_chunks)} chunks in ChromaDB")

Total chunks: 601


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored 601 chunks in ChromaDB
